In [3]:
# ============================================================
# CELL 1: Imports + Modell-Konfiguration
# ============================================================
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
from pathlib import Path
from transformers import CLIPProcessor, CLIPModel
import open_clip
import mobileclip
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import joblib

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Gerät: {device}")

MOBILECLIP_WEIGHTS = r"C:\Users\Lukas\masterthesis\thesis_2026\data\weights\mobileclip_weights\mobileclip_s1.pt"

MODEL_CONFIGS = [
    {"name": "CLIP-ViT-B32",   "type": "hf",        "model_id": "openai/clip-vit-base-patch32"},
    {"name": "CLIP-ViT-L14",   "type": "hf",        "model_id": "openai/clip-vit-large-patch14"},
    {"name": "EVA01-CLIP-g14", "type": "open_clip", "model_id": "EVA01-g-14", "pretrained": "laion400m_s11b_b41k"},
    {"name": "MobileCLIP-S1",  "type": "mobileclip", "model_id": "mobileclip_s1", "pretrained": MOBILECLIP_WEIGHTS},
]

def load_model(config):
    if config["type"] == "hf":
        processor = CLIPProcessor.from_pretrained(config["model_id"])
        model = CLIPModel.from_pretrained(config["model_id"]).to(device)
        model.eval()
        return {"type": "hf", "model": model, "processor": processor}
    elif config["type"] == "open_clip":
        model, _, preprocess = open_clip.create_model_and_transforms(
            config["model_id"], pretrained=config["pretrained"]
        )
        model = model.to(device)
        model.eval()
        return {"type": "open_clip", "model": model, "preprocess": preprocess}
    elif config["type"] == "mobileclip":
        model, _, preprocess = mobileclip.create_model_and_transforms(
            config["model_id"], pretrained=config["pretrained"], device=device
        )
        model.eval()
        return {"type": "mobileclip", "model": model, "preprocess": preprocess}

def encode_image(mb, img):
    if mb["type"] == "hf":
        inputs = mb["processor"](images=img, return_tensors="pt").to(device)
        with torch.no_grad():
            out = mb["model"].get_image_features(**inputs)
        feat = out if isinstance(out, torch.Tensor) else out.pooler_output
    else:  # open_clip oder mobileclip
        tensor = mb["preprocess"](img).unsqueeze(0).to(device)
        with torch.no_grad():
            feat = mb["model"].encode_image(tensor)
    feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.cpu().numpy()[0]


# ============================================================
# CELL 2: Gold Standard laden (CSV)
# ============================================================
GOLD_STANDARD_CSV = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Human_Non_Human\23062026_annotations_human_non_human.csv"
IMAGE_DIR = Path(r"C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Human_Non_Human\18062026_Random_sample_for_annotation_human_non_human\random_sample_images_human_non_human")

df_gold = pd.read_csv(GOLD_STANDARD_CSV)

def norm(x):
    if x is None or pd.isna(x):
        return None
    return x.strip().lower().replace("-", "_")

filenames, labels = [], []
for _, row in df_gold.iterrows():
    fname = row.iloc[0]              # erste Spalte: filename
    gold = norm(row.iloc[8])         # neunte Spalte (Index 8): gold_standard_final
    if fname is None or gold is None:
        continue
    filenames.append(fname)
    labels.append(gold)

print(f"Geladen: {len(filenames)} Bilder mit Gold-Standard-Label")


# ============================================================
# CELL 3: Hilfsfunktion zum sicheren Öffnen
# ============================================================
def safe_open(path):
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError) as e:
        print(f"  ⚠ Skipped ({os.path.basename(path)}): {e}")
        return None


# ============================================================
# CELL 4: Hauptschleife — pro Modell Linear Probe trainieren + evaluieren
# ============================================================
output_dir = r"C:\Users\Lukas\masterthesis\thesis_2026\data\LinearProbe_AllModels_Comparison"
os.makedirs(output_dir, exist_ok=True)

all_results = []

for config in MODEL_CONFIGS:
    model_name = config["name"]
    print(f"\n{'='*60}\nModell: {model_name}\n{'='*60}")

    model_bundle = load_model(config)

    # --- Embeddings extrahieren ---
    X, y, missing = [], [], []
    for fname, label in zip(filenames, labels):
        img_path = IMAGE_DIR / fname
        if not img_path.exists():
            missing.append(fname)
            continue
        img = safe_open(img_path)
        if img is None:
            continue
        emb = encode_image(model_bundle, img)
        X.append(emb)
        y.append(label)

    if missing:
        print(f"  WARNUNG: {len(missing)} Bilder nicht gefunden")

    X = np.array(X)
    y = np.array(y)
    print(f"  Embeddings extrahiert für {len(X)} Bilder (Dim={X.shape[1]})")

    np.save(os.path.join(output_dir, f"embeddings_{model_name}.npy"), X)

    # --- Linear Probe + 5-fold CV ---
    clf = LogisticRegression(max_iter=1000, C=0.5, class_weight="balanced")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    auc_scores = cross_val_score(clf, X, y, cv=skf, scoring="roc_auc")
    y_pred = cross_val_predict(clf, X, y, cv=skf)

    report = classification_report(y, y_pred, output_dict=True)
    cm = confusion_matrix(y, y_pred, labels=["human", "non_human"])

    print(f"  AUC (5-fold CV): {auc_scores.mean():.3f} +/- {auc_scores.std():.3f}")
    print(classification_report(y, y_pred))
    print("  Confusion Matrix (human, non_human):")
    print(cm)

    # --- Finales Modell trainieren + speichern ---
    clf.fit(X, y)
    joblib.dump(clf, os.path.join(output_dir, f"linear_probe_{model_name}.joblib"))

    # --- Ergebnisse sammeln ---
    all_results.append({
        "Modell": model_name,
        "Embedding_Dim": X.shape[1],
        "AUC_mean": round(auc_scores.mean(), 4),
        "AUC_std": round(auc_scores.std(), 4),
        "Accuracy": round(report["accuracy"], 4),
        "Precision_human": round(report["human"]["precision"], 4),
        "Recall_human": round(report["human"]["recall"], 4),
        "F1_human": round(report["human"]["f1-score"], 4),
        "Precision_nonhuman": round(report["non_human"]["precision"], 4),
        "Recall_nonhuman": round(report["non_human"]["recall"], 4),
        "F1_nonhuman": round(report["non_human"]["f1-score"], 4),
        "FP_human_FromNonHuman": int(cm[1][0]),  # non_human fälschlich als human
        "FN_human_ToNonHuman": int(cm[0][1]),    # human fälschlich als non_human
    })

    del model_bundle
    torch.cuda.empty_cache()


# ============================================================
# CELL 5: Gesamtvergleich aller Modelle
# ============================================================
df_comparison = pd.DataFrame(all_results).sort_values("AUC_mean", ascending=False)
print("\n" + "="*60)
print("GESAMTVERGLEICH — Linear Probe pro Modell")
print("="*60)
print(df_comparison.to_string(index=False))

comparison_csv = os.path.join(output_dir, "linearprobe_model_comparison.csv")
df_comparison.to_csv(comparison_csv, index=False)
print(f"\nGespeichert: {comparison_csv}")

Gerät: cuda
Geladen: 298 Bilder mit Gold-Standard-Label

Modell: CLIP-ViT-B32


Loading weights: 100%|████████████████████████████████████████████████████████████| 398/398 [00:00<00:00, 15410.84it/s]


  Embeddings extrahiert für 298 Bilder (Dim=512)
  AUC (5-fold CV): 0.989 +/- 0.009
              precision    recall  f1-score   support

       human       0.84      0.98      0.90        92
   non_human       0.99      0.92      0.95       206

    accuracy                           0.94       298
   macro avg       0.92      0.95      0.93       298
weighted avg       0.94      0.94      0.94       298

  Confusion Matrix (human, non_human):
[[ 90   2]
 [ 17 189]]

Modell: CLIP-ViT-L14


Loading weights: 100%|█████████████████████████████████████████████████████████████| 590/590 [00:00<00:00, 3485.20it/s]


  Embeddings extrahiert für 298 Bilder (Dim=768)
  AUC (5-fold CV): 0.991 +/- 0.007
              precision    recall  f1-score   support

       human       0.86      0.98      0.91        92
   non_human       0.99      0.93      0.96       206

    accuracy                           0.94       298
   macro avg       0.92      0.95      0.94       298
weighted avg       0.95      0.94      0.94       298

  Confusion Matrix (human, non_human):
[[ 90   2]
 [ 15 191]]

Modell: EVA01-CLIP-g14
  Embeddings extrahiert für 298 Bilder (Dim=1024)
  AUC (5-fold CV): 0.987 +/- 0.015
              precision    recall  f1-score   support

       human       0.82      0.98      0.89        92
   non_human       0.99      0.90      0.94       206

    accuracy                           0.93       298
   macro avg       0.90      0.94      0.92       298
weighted avg       0.94      0.93      0.93       298

  Confusion Matrix (human, non_human):
[[ 90   2]
 [ 20 186]]

Modell: MobileCLIP-S1
  Embe